# 📘 Notebook 04: Formal Languages and Automata

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module B: The Structure of Language · Notebook 1 of 3 in this module · (4 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Define an **alphabet**, a **string** and a **language** in the formal sense
- Explain what a **finite automaton** is, and implement a **DFA** in a few lines of Python
- Trace an automaton by hand on an input string
- Simulate a **non-deterministic** automaton (NFA) and explain how it relates to a DFA
- State the connection between finite automata and **regular expressions**, and test it
- Explain why some languages are **beyond** any finite automaton, and place them in the **Chomsky hierarchy**

> **Prerequisites:** Notebook 02 (regular expressions). Python dictionaries and sets.
>
> 🔭 **Why this notebook matters:** Notebook 02 ended with a puzzle: no regular expression can check that brackets are balanced. To see why, we need to look at the machine that sits behind every regular expression. That machine is one of the simplest models of computation there is, and understanding exactly what it can and cannot do tells us what kind of tool human language requires.

> 📦 **Libraries used in this notebook.** None. Everything here is written with Python itself, so there is nothing to install.

## 1. Languages as sets of strings

### Intuition first
In everyday speech a language is something people speak. In this notebook the word has a narrower and more precise meaning. Imagine sorting every possible sequence of symbols into two piles, *"belongs"* and *"does not belong"*. The pile of sequences that belong is the language. Nothing is said about meaning. A language is defined purely by **membership**.

### Formal definition
- An **alphabet**, written $\Sigma$, is a finite set of symbols. Example: $\Sigma = \{a, b\}$.
- A **string** over $\Sigma$ is a finite sequence of symbols from it: `abba`. The **empty string**, with no symbols at all, is written $\varepsilon$.
- $\Sigma^*$ is the set of **all** strings over $\Sigma$, of every length. It is infinite.
- A **language** is any subset of $\Sigma^*$.

Let us generate the beginning of $\Sigma^*$ for the alphabet $\{a, b\}$.

In [ ]:
from itertools import product

def all_strings(alphabet, max_length):
    result = [""]                                   # the empty string
    for length in range(1, max_length + 1):
        for symbols in product(alphabet, repeat=length):
            result.append("".join(symbols))
    return result

strings = all_strings("ab", 3)
print(strings)
print(len(strings), "strings of length 0 to 3")

A language picks some of these and leaves the rest. Here are two languages over the same alphabet, each defined by a rule.

In [ ]:
even_number_of_a = []
ends_with_ab = []

for string in strings:
    if string.count("a") % 2 == 0:
        even_number_of_a.append(string)
    if string.endswith("ab"):
        ends_with_ab.append(string)

print("Even number of a:", even_number_of_a)
print("Ends with ab:    ", ends_with_ab)

> 🧠 The alphabet does not have to be letters. If the symbols are English **words**, then a string is a sequence of words, and "the set of all grammatical English sentences" is a language in exactly this sense. That is the bridge from this abstract notebook to the rest of the course. The question *"is this a sentence of English?"* is a membership question.

## 2. The finite automaton

### Intuition first
Think of a turnstile at a metro station. It is always in one of two conditions, *locked* or *unlocked*. Insert a ticket and it unlocks. Push through and it locks again. Push while it is locked and nothing happens. The turnstile has no memory of how many people passed today. All it knows is which of its two **states** it is in, and each event moves it from one state to another.

A **finite automaton** is exactly that: a machine with a fixed number of states that reads a string one symbol at a time and changes state after each symbol. When the string ends, the machine is either in a state marked "accept" or it is not. That is its entire answer.

### Formal definition
A **deterministic finite automaton (DFA)** consists of five things:

1. a finite set of **states**, $Q$
2. an **alphabet**, $\Sigma$
3. a **transition function** $\delta$: for each state and each symbol, the one state to move to
4. a **start state**, $q_0$
5. a set of **accepting states**, $F$

The DFA **accepts** a string if, starting from $q_0$ and following $\delta$ for every symbol, it finishes in a state of $F$. The set of all strings it accepts is the language it **recognises**.

Here is a DFA for the language *"an even number of a"*. It needs two states, because the only thing worth remembering is whether the number of `a` seen so far is even or odd.

| state | on `a` go to | on `b` go to |
|---|---|---|
| **even** (start, accepting) | odd | even |
| **odd** | even | odd |

In Python, a dictionary holds the table.

In [ ]:
even_a = {
    "start": "even",
    "accepting": {"even"},
    "transitions": {
        ("even", "a"): "odd",
        ("even", "b"): "even",
        ("odd", "a"): "even",
        ("odd", "b"): "odd",
    },
}

def run_dfa(dfa, text, trace=False):
    state = dfa["start"]
    for symbol in text:
        if (state, symbol) not in dfa["transitions"]:
            return False                      # no rule for this symbol: reject
        next_state = dfa["transitions"][(state, symbol)]
        if trace:
            print(f"  in state {state:<6} read {symbol!r} -> go to {next_state}")
        state = next_state
    return state in dfa["accepting"]

for text in ["", "b", "a", "aa", "ab", "aba", "abab", "baaab"]:
    print(repr(text), run_dfa(even_a, text))

### Tracing it by hand
Follow the machine on the string `abab`, one symbol at a time.

In [ ]:
print(run_dfa(even_a, "abab", trace=True))

The whole interpreter is seven lines long. It never looks back at earlier symbols and never looks ahead. Whatever it needs to know about the past must be summarised in the name of the current state.

Does the machine really recognise the language we intended? We never take a claim on faith. Let us compare it with the direct definition on **every** string up to length 10.

In [ ]:
tests = all_strings("ab", 10)

disagreements = 0
for text in tests:
    if run_dfa(even_a, text) != (text.count("a") % 2 == 0):
        disagreements += 1

print(f"Checked {len(tests):,} strings. Disagreements: {disagreements}")

## 3. An automaton for a practical job

The alphabet $\{a, b\}$ is for learning. The same machine can validate real input. Let us accept simplified e-mail addresses of this shape:

- the name starts with a letter and continues with letters or digits
- then exactly one `@`
- then one or more parts made of letters, separated by single dots, with at least one dot

Listing a transition for every letter of the alphabet would be tedious, so we first map each character to a **class**: `letter`, `digit`, `@`, `.` or `other`. The automaton then works on classes.

In [ ]:
def character_class(character):
    if character.isalpha():
        return "letter"
    if character.isdigit():
        return "digit"
    if character in "@.":
        return character
    return "other"

email_dfa = {
    "start": "start",
    "accepting": {"ending"},
    "transitions": {
        ("start", "letter"): "name",
        ("name", "letter"): "name",
        ("name", "digit"): "name",
        ("name", "@"): "at",
        ("at", "letter"): "domain",
        ("domain", "letter"): "domain",
        ("domain", "."): "dot",
        ("dot", "letter"): "ending",
        ("ending", "letter"): "ending",
        ("ending", "."): "dot",
    },
}

def is_email(text):
    classes = []
    for character in text:
        classes.append(character_class(character))
    return run_dfa(email_dfa, classes)

for text in ["anna@uni.gr", "nikos7@mail.example.com", "7nikos@mail.com", "anna@uni", "anna@@uni.gr", "anna@uni..gr", "anna@uni.gr."]:
    print(f"{text:<26} {is_email(text)}")

Read the transition table as a story. From `name` the only way forward is an `@`. After the `@` a letter is required. A dot must be followed by a letter, which is why `anna@uni..gr` and `anna@uni.gr.` are rejected. And the only accepting state is `ending`, which can be reached only after a dot, so `anna@uni` fails.

> ⚠️ **Common pitfalls**
>
> - Forgetting that **missing transitions mean rejection**. A complete DFA has a rule for every state and every symbol. In practice we leave out the hopeless ones and treat "no rule" as "reject", as `run_dfa` does.
> - Marking the wrong states as accepting. The automaton accepts only if the input **ends** in an accepting state. Passing through one on the way does not count.
> - Trying to make a state remember "how many". A state can remember *even or odd*, or *at most three*, because those are finitely many possibilities. It cannot remember an unlimited count.

## 4. Non-determinism

### Intuition first
In a DFA every step is forced: one state, one symbol, one destination. A **non-deterministic** automaton may have **several** possible moves for the same symbol, or none. You can picture it as a machine that makes a lucky guess at every fork, or, more usefully, as a machine that follows **all** possibilities at once and accepts if **any** of them ends in an accepting state.

Take the language *"strings that end with ab"*. The difficulty for a machine reading left to right is that it does not know which `a` is the second-to-last symbol. A non-deterministic machine simply guesses.

| state | on `a` | on `b` |
|---|---|---|
| **q0** (start) | q0 **or** q1 | q0 |
| **q1** | nothing | q2 |
| **q2** (accepting) | nothing | nothing |

In `q0` the machine loops on every symbol. Whenever it reads an `a`, it may also bet that this is the beginning of the final `ab` and move to `q1`.

### Formal definition
A **non-deterministic finite automaton (NFA)** is defined like a DFA, except that the transition function returns a **set** of states. It accepts a string if **at least one** sequence of choices leads to an accepting state.

To simulate it, we keep track of the set of all states it could be in.

In [ ]:
ends_ab = {
    "start": "q0",
    "accepting": {"q2"},
    "transitions": {
        ("q0", "a"): {"q0", "q1"},
        ("q0", "b"): {"q0"},
        ("q1", "b"): {"q2"},
    },
}

def run_nfa(nfa, text, trace=False):
    current = {nfa["start"]}
    for symbol in text:
        following = set()
        for state in current:
            following |= nfa["transitions"].get((state, symbol), set())
        if trace:
            print(f"  read {symbol!r}: {sorted(current)} -> {sorted(following)}")
        current = following
    return len(current & nfa["accepting"]) > 0

print(run_nfa(ends_ab, "abaab", trace=True))

After the last symbol the set of possible states contains `q2`, so the string is accepted. Watch what happened along the way: every `a` opened a new bet in `q1`, and every bet that was not followed by a final `b` quietly died.

### NFAs are no more powerful than DFAs
Look at what `run_nfa` actually does. At each step it is in one **set of states**, and the next set is completely determined by the current set and the symbol. That is a deterministic machine whose states are sets of NFA states. This observation is a famous theorem:

> Every NFA can be converted into a DFA that recognises exactly the same language.

Non-determinism buys convenience, not power. The NFA above has 3 states and is easy to design. The price of converting is size: an NFA with $n$ states can require up to $2^n$ states as a DFA.

## 5. Regular expressions are automata in disguise

The language *"ends with ab"* can also be written as the regular expression `[ab]*ab`. That is no coincidence.

> **Kleene's theorem.** A language can be described by a regular expression **if and only if** it can be recognised by a finite automaton.

The languages in this class are called the **regular languages**. A regular-expression engine works by turning your pattern into an automaton and running it, much as we just did. Let us test the claim on our example, on every string up to length 10.

In [ ]:
import re

disagreements = 0
for text in tests:
    by_automaton = run_nfa(ends_ab, text)
    by_regex = re.fullmatch(r"[ab]*ab", text) is not None
    if by_automaton != by_regex:
        disagreements += 1

print(f"Checked {len(tests):,} strings. Disagreements: {disagreements}")

Three notations, one class of languages: regular expressions, DFAs and NFAs describe exactly the same things. Each piece of regex syntax corresponds to a way of wiring states together. `ab` means one state after another, `a|b` means a fork, and `a*` means a loop.

## 6. What finite automata cannot do

### Intuition first
The word *finite* is the whole story. A finite automaton with 10 states can be in only 10 different situations. It has no notebook, no counter and no stack. Everything it knows about the input so far is which state it is in.

Now consider the language $a^n b^n$: some number of `a`, followed by **the same number** of `b`. So `ab`, `aabb` and `aaabbb` belong, and `aab` does not. To check a string, the machine must remember exactly how many `a` it has read. That number can be arbitrarily large, and the machine has only finitely many states.

Let us try anyway. The function below builds a DFA that counts correctly up to a chosen limit, using one state for every count.

In [ ]:
def build_counting_dfa(limit):
    transitions = {}
    for count in range(limit):
        transitions[(f"a{count}", "a")] = f"a{count + 1}"     # one more a seen
    for count in range(1, limit + 1):
        transitions[(f"a{count}", "b")] = f"b{count - 1}"     # first b: one fewer b owed
    for owed in range(1, limit):
        transitions[(f"b{owed}", "b")] = f"b{owed - 1}"       # each further b
    return {"start": "a0", "accepting": {"a0", "b0"}, "transitions": transitions}

dfa = build_counting_dfa(3)
states = set()
for (state, symbol), target in dfa["transitions"].items():
    states.add(state)
    states.add(target)
print(len(states), "states:", sorted(states))

for n in range(6):
    text = "a" * n + "b" * n
    print(f"n = {n}  {text:<11} {run_dfa(dfa, text)}")

With a limit of 3 the machine needs 7 states, and it is correct up to `aaabbb`. At $n = 4$ it runs out of states and rejects a string that belongs to the language. We can raise the limit, and the failure simply moves further out.

In [ ]:
for limit in [3, 10, 100]:
    dfa = build_counting_dfa(limit)
    first_failure = None
    for n in range(limit + 5):
        if not run_dfa(dfa, "a" * n + "b" * n):
            first_failure = n
            break
    print(f"limit {limit:>3}: {2 * limit + 1:>3} states, first wrong answer at n = {first_failure}")

However many states we add, there is always a longer string that defeats the machine. This can be proved rigorously: **$a^n b^n$ is not a regular language**. No finite automaton recognises it, and therefore no regular expression describes it. This is the precise reason behind the bracket puzzle of Notebook 02, since `((()))` is just $a^n b^n$ written with different symbols.

### One extra ingredient is enough
The fix is tiny. Give the program a single unbounded counter.

In [ ]:
def is_anbn(text):
    count = 0
    seen_b = False
    for symbol in text:
        if symbol == "a":
            if seen_b:
                return False          # an a after a b: wrong shape
            count += 1
        elif symbol == "b":
            seen_b = True
            count -= 1
            if count < 0:
                return False          # more b than a
        else:
            return False
    return count == 0

for text in ["", "ab", "aaaabbbb", "aab", "abab", "a" * 1000 + "b" * 1000]:
    shown = text if len(text) < 12 else f"a x 1000, b x 1000"
    print(f"{shown:<20} {is_anbn(text)}")

A machine with unbounded memory of this "last in, first out" kind is called a **pushdown automaton**, and the languages it recognises are the **context-free languages**. They are the subject of the next notebook.

## 7. The Chomsky hierarchy, and where human language sits

In the 1950s the linguist Noam Chomsky arranged formal languages into four nested classes, each recognised by a more powerful machine than the last.

| Type | Languages | Machine that recognises them | Memory | Example |
|---|---|---|---|---|
| 3 | **regular** | finite automaton | none beyond the state | $a^*b^*$ |
| 2 | **context-free** | pushdown automaton | a stack | $a^n b^n$ |
| 1 | **context-sensitive** | linear bounded automaton | tape as long as the input | $a^n b^n c^n$ |
| 0 | **unrestricted** | Turing machine | unlimited tape | anything computable |

Every regular language is also context-free, every context-free language is also context-sensitive, and so on outwards.

### Why this matters for natural language
English lets you put a clause inside a clause:

- *The rat died.*
- *The rat **the cat chased** died.*
- *The rat **the cat the dog bit chased** died.*

Each noun opened on the left must be closed by its verb on the right, in reverse order. The pattern is *noun noun noun ... verb verb verb*, with matching counts: the shape of $a^n b^n$. So the grammar of English is **not regular**. No finite automaton, and no regular expression, can capture it in full.

This is the formal reason why Notebook 02 was a notebook about cleaning text, and not about understanding it. Regular expressions are the right tool for the **shape of words and tokens**. For the **structure of sentences** we need at least the next level of the hierarchy.

> 🧠 People struggle with the third sentence above, and a fourth level of nesting is practically unreadable. Human memory is finite too. This is a recurring theme: the grammar of a language allows far more than its speakers ever use, and a model of real usage does not always need the full power of the grammar. We will return to this idea when we meet statistical models in Module C.

---
## ✏️ Exercises

### Exercise 1 (Contains "aba")
Design a DFA over the alphabet $\{a, b\}$ that accepts exactly the strings that **contain** `aba` somewhere. Write it as a dictionary for `run_dfa`, and check it against Python's `"aba" in text` on all strings up to length 10. (Hint: four states, each remembering how much of `aba` has just been seen.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
contains_aba = {
    "start": "nothing",
    "accepting": {"found"},
    "transitions": {
        ("nothing", "a"): "a",    ("nothing", "b"): "nothing",
        ("a", "a"): "a",          ("a", "b"): "ab",
        ("ab", "a"): "found",     ("ab", "b"): "nothing",
        ("found", "a"): "found",  ("found", "b"): "found",
    },
}

wrong = 0
for text in all_strings("ab", 10):
    if run_dfa(contains_aba, text) != ("aba" in text):
        wrong += 1
print("Disagreements:", wrong)
```

*Note the transition from state "a" on another a: it stays in "a", because the new a could itself be the start of "aba". Once "found" is reached the machine never leaves it.*
</details>

### Exercise 2 (Divisible by three)
A string of `0` and `1` can be read as a binary number. Design a DFA that accepts exactly the binary numbers **divisible by 3**. (Hint: use three states for the remainder so far. Reading a new bit turns a number $x$ into $2x + \text{bit}$.) Check it against `int(text, 2) % 3 == 0` for all binary strings of length 1 to 10.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
transitions = {}
for remainder in [0, 1, 2]:
    for bit in [0, 1]:
        transitions[(f"r{remainder}", str(bit))] = f"r{(2 * remainder + bit) % 3}"

divisible_by_3 = {"start": "r0", "accepting": {"r0"}, "transitions": transitions}

wrong = 0
checked = 0
for text in all_strings("01", 10):
    if text == "":
        continue
    checked += 1
    if run_dfa(divisible_by_3, text) != (int(text, 2) % 3 == 0):
        wrong += 1
print(f"Checked {checked:,} numbers. Disagreements: {wrong}")
```

*The numbers grow without limit, but the machine only has to remember the remainder, and there are just three possible remainders. Deciding what small summary of the past is enough is the whole art of designing an automaton.*
</details>

### Exercise 3 (The third symbol from the end)
Write an **NFA** for the language of strings over $\{a, b\}$ whose **third symbol from the end** is `a`, and simulate it with `run_nfa`. Check it against the regular expression `[ab]*a[ab]{2}` on all strings up to length 10.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
third_from_end = {
    "start": "q0",
    "accepting": {"q3"},
    "transitions": {
        ("q0", "a"): {"q0", "q1"},
        ("q0", "b"): {"q0"},
        ("q1", "a"): {"q2"}, ("q1", "b"): {"q2"},
        ("q2", "a"): {"q3"}, ("q2", "b"): {"q3"},
    },
}

wrong = 0
for text in all_strings("ab", 10):
    if run_nfa(third_from_end, text) != (re.fullmatch(r"[ab]*a[ab]{2}", text) is not None):
        wrong += 1
print("Disagreements:", wrong)
```

*The NFA guesses which a is the third from the end and then simply counts two more symbols. It needs 4 states. An equivalent DFA must remember the last three symbols it read, which takes 8 states. For "the tenth symbol from the end" the NFA needs 11 states and the DFA 1,024.*
</details>

### Exercise 4 (A clock)
Build a DFA that accepts valid 24-hour times written as `HH:MM`, from `00:00` to `23:59`. Build the transition dictionary with loops over the digits. Test it on `"09:30"`, `"23:59"`, `"24:00"`, `"12:60"`, `"7:15"` and `"12:5"`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
transitions = {}
for digit in "01":
    transitions[("start", digit)] = "hour_low"       # first digit 0 or 1: any second digit
transitions[("start", "2")] = "hour_high"            # first digit 2: second digit only 0 to 3
for digit in "0123456789":
    transitions[("hour_low", digit)] = "hours"
for digit in "0123":
    transitions[("hour_high", digit)] = "hours"
transitions[("hours", ":")] = "colon"
for digit in "012345":
    transitions[("colon", digit)] = "minute_tens"
for digit in "0123456789":
    transitions[("minute_tens", digit)] = "done"

clock = {"start": "start", "accepting": {"done"}, "transitions": transitions}

for text in ["09:30", "23:59", "24:00", "12:60", "7:15", "12:5"]:
    print(text, run_dfa(clock, text))
```

*Two different states after the first digit are needed, because what is allowed next depends on it. This is the same language as the regular expression `([01]\d|2[0-3]):[0-5]\d`: the "or" in the pattern is the fork in the automaton.*
</details>

### Exercise 5 (Two kinds of brackets, a little harder)
Write `balanced(text)` that returns `True` when round and square brackets are correctly nested: `"([])"` and `"()[]"` are fine, `"(]"` and `"([)]"` are not. A single counter is no longer enough. Use a Python list as a **stack**: push every opening bracket, and for every closing bracket check that it matches the one on top.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def balanced(text):
    partner = {")": "(", "]": "["}
    stack = []
    for symbol in text:
        if symbol in "([":
            stack.append(symbol)
        elif symbol in ")]":
            if not stack or stack.pop() != partner[symbol]:
                return False
    return len(stack) == 0

for text in ["([])", "()[]", "(]", "([)]", "((", "[()[]]"]:
    print(f"{text:<8} {balanced(text)}")
```

*This is a pushdown automaton written as a Python function: a finite set of rules plus a stack. The stack remembers not only how many brackets are open but which kind and in what order. It is exactly the memory needed for the nested clauses of a sentence.*
</details>

## 🔑 Key takeaways

- Formally, a **language** is a set of strings over an alphabet. Asking whether a sentence is grammatical is a membership question.
- A **finite automaton** reads one symbol at a time and remembers only its current **state**.
- A **DFA** has exactly one move per state and symbol. An **NFA** may have several, and is simulated by tracking a set of states. They recognise the same languages.
- **Regular expressions and finite automata are equivalent**: both describe exactly the **regular languages**.
- A finite automaton cannot count without limit, so it cannot recognise $a^n b^n$ or balanced brackets.
- The **Chomsky hierarchy** ranks languages by the memory needed to recognise them: regular, context-free, context-sensitive, unrestricted.
- The nested structure of human sentences is **not regular**. It needs at least a stack.

---
**Next:** *Notebook 05: Grammars and Parsing*, where we climb one level of the hierarchy and write rules that describe the structure of whole sentences.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*